# Three-Dimensional Ensemble-Variational (3DEnVar) Tutorial
---

## Prerequisites

<u>This tutorial requires completion of the **0.qg_tutorial_start** tutorial</u>. We will use the truth, background, and observations generated from that tutorial. 

Although this tutorial can be completed independently of other DA solver tutorials, it is also strongly recommended that you first complete and understand  **1.qg_3Dvar_tutorial** and **3.qg_LETKF_tutorial**, since the 3DEnVar is a hybridized combination of variational and ensemble DA approaches and relies upon the user being familiar with these two approaches already. 

## 1. Introduction

In the previous 3D-Var tutorial, we learned how to assimilate observations by minimizing a cost function that penalizes departures from both the background state and the observations. The background error covariance matrix $\mathbf{B}$ plays a critical role in this process by determining how information from observations spreads spatially and across variables. However, the 3D-Var method typically uses a static background error covariance, meaning that $\mathbf{B}$ does not change with the flow of the day. This static covariance is often constructed from climatological statistics and lacks the ability to capture flow-dependent error structures that vary with the current atmospheric state. Ensemble data assimilation methods, such as the Ensemble Kalman Filter (EnKF), address this by estimating $\mathbf{B}$ from an ensemble of forecasts. The ensemble provides a flow-dependent estimate of forecast uncertainty, which can be used to construct a dynamic background error covariance.

Three-Dimensional Ensemble-Variational data assimilation (3DEnVar) combines the strengths of both approaches. It retains the stable, well-optimized variational framework of 3D-Var with benefits such as, e.g., enforcing dynamic constraints within the cost function, while also incorporating flow-dependent error information from an ensemble. This hybrid approach has become a cornerstone of operational numerical weather prediction systems.

In this tutorial, we will explore how 3DEnVar extends the 3D-Var framework by blending static and ensemble-derived background error covariances. We will derive the mathematical formulation, implement the method in JEDI using the QG model, and examine the impact of different weighting strategies on the analysis.

## 2. Two mathematical formulations of 3DEnVar

#### 2.1 Formulation 1: Ensemble covariances in the B-matrix (JEDI used method)

In 3DEnVar, we replace the static background error covariance $\mathbf{B}$ of the 3D-Var cost function with a hybrid covariance that combines static and ensemble-derived components:

$$
\mathbf{B}_{hybrid} = \beta \mathbf{B}_{static} + (1 - \beta) (\mathcal{L} \circ \mathbf{B}_{ens})
$$

where:
- $\mathbf{B}_{static}$ is the static background error covariance (as used in 3D-Var)
- $\mathbf{B}_{ens}$ is the ensemble-derived background error covariance
- $\beta$ is the weighting coefficient
- $\mathcal{L} \circ \mathbf{B}_{ens}$ denotes the Schur (element-wise) product of the ensemble covariance with a localization matrix $\mathcal{L}$

Given ${N}$ ensemble members with perturbations $\mathbf{x}'_{i} = \mathbf{x}_i - \bar{\mathbf{x}}$, the ensemble-based covariance is:

$$
\mathbf{B}_{ens} = \frac{1}{N-1} \sum_{i=1}^{N} \mathbf{x}'_i (\mathbf{x}'_i)^T
$$

The weighting coefficient $\beta$ controls the relative contributions of the static and ensemble components. When $\beta = 1$, the method reduces to standard 3D-Var. When $\beta = 0$, the method uses only the ensemble covariances.

The **localization $\mathcal{L}$** is necessary because ensemble covariances estimated from a finite ensemble contain sampling errors, particularly for long-range correlations. Localization dampens spurious correlations at large distances, improving the quality of the ensemble covariance estimate.


#### 2.2 Formulation 2: Extended alpha control variable (e.g., GSI used method; Wang 2010 MWR)

Specifically, the analysis increment can be written schematically as
$$
\mathbf{x}' = \mathbf{x}'_{1} + \sum_{k=1}^{K} (\mathbf{\alpha}_{k} \circ \mathbf{x}^{e}_{k} )
$$
where:
- $\mathbf{x}_{1}$ is the increment associated with the static background covariance.
- $\mathbf{\alpha}_{k}$ denotes the extended control variables for each ensemble member.
The second term of the equation above represents a local linear combination of ensemble perturbations.

The cost function is augmented with a penalty on $\boldsymbol{\alpha}$
$$
{J}(\mathbf{x}'_{1}, \boldsymbol{\alpha}) = \beta\frac{1}{2}(\mathbf{x}'_{1})^T\mathbf{B}^{-1}_{static}(\mathbf{x}'_{1}) + (1 - \beta)\frac{1}{2}(\boldsymbol{\alpha})^T\mathbf{A}^{-1}(\boldsymbol{\alpha}) + \frac{1}{2}(\mathbf{y}^{o'} - \mathbf{H}\mathbf{x}')^T\mathbf{R}^{-1}(\mathbf{y}^{o'} - \mathbf{H}\mathbf{x}')
$$

- $\mathbf{A}$ determines the covariance localization on the ensemble covariance.

These differently proposed 3DEnVar schemes in **2.1** and **2.2** have been proven to be equivalent (Wang et al. 2007 MWR).

In many implementations, 3DEnVar is used in a hybrid configuration, in which flow-dependent and static covariances are combined. Next, we will perform single- and multi-observation experiments to examine the effects of the combination.

### Export variables and verify executables/directories

In [1]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU

Verify the executables and output directories:

In [2]:
ls -RCd --color $JEDI_BIN/qg_4dvar.x         
ls -RCd --color $JEDI_EDU/qgEnVar/output/*

/home/nonroot/build/bin/qg_4dvar.x
/home/nonroot/shared/EDU/qgEnVar/output/exp_b10e90
/home/nonroot/shared/EDU/qgEnVar/output/exp_b90e10
/home/nonroot/shared/EDU/qgEnVar/output/exp_default
/home/nonroot/shared/EDU/qgEnVar/output/exp_mult_b10e90
/home/nonroot/shared/EDU/qgEnVar/output/exp_mult_b50e50
/home/nonroot/shared/EDU/qgEnVar/output/exp_mult_b90e10


### Table of 3DEnVar experiments

| Experiment Name | 3DEnVar yaml file | Description |
| --- | --- | --- |
| exp_default | 3dvar_hybrid.yaml | Default single-observation experiment ($\alpha=0.5$)
| exp_b10e90 | 3dvar_hybrid_b10e90.yaml | Single-ob experiment, weighted towards ensemble ($\alpha=0.1$)
| exp_b90e10 | 3dvar_hybrid_b90e10.yaml | Single-ob experiment, weighted towards static-B ($\alpha=0.9$)
| exp_mult_b50e50  | 3dvar_hybrid_mult_b50e50.yaml | Assimilates 100 observations, $\alpha=0.5$
| exp_mult_b10e90  | 3dvar_hybrid_mult_b10e90.yaml | 100-obs DA, weighted towards ensemble ($\alpha=0.1$)
| exp_mult_b90e10  | 3dvar_hybrid_mult_b90e10.yaml | 100-obs DA, weighted towards static-B ($\alpha=0.9$)



## Experiments 1-3: 3DEnVar Single-Observation Experiments with Varying Weights 
***

### Step 1: Verify truth, observations, and background files exist

The truth, observations, and ensemble files were produced from the `qgstart` tutorial. **Note that the 20-member ensemble is the same that was used for the LETKF tutorial previously.**

In [3]:
ls $JEDI_EDU/qgstart/output/truth/*D.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Truth files don't exist! Please run the 0.qg_tutorial_start tutorial first to generate the truth simulation"
fi

/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P100D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P101D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P102D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P103D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P104D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P105D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P106D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P107D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P108D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P109D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P10D.nc
/home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T

In [4]:
# Verify that ensemble files were produced
ls -S $JEDI_EDU/qgstart/output/bg/bkgd*P1D.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Background files don't exist! Please see the 0.qg_tutorial_start tutorial to generate the background ensemble"
fi

/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ensmean.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.1.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.10.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.11.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.12.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.13.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.14.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.15.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.16.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.17.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.18.2009-12-30T00:00:00Z.P1D.nc
/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.19.2009-12-30T00:00:00Z.P1D.nc
/hom

The above command should list out 20 ensemble members (starting with `bkgd.ens.X...` for different member numbers of `X`) as well as a single `bkgd.fc...` file that will be used as the control member for the 3DEnVar analysis.

In [5]:
# Verify that obs file exists
ls -S $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Obs files doesn't exist! Please see the 0.qg_tutorial_start tutorial to generate the observations"
fi

/home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc


### Step 2: Run 3DEnVar experiments

#### Experiment 1: Default (static 0.5, ensemble 0.5)

For the default experiment, we will configure the weights to be equal between static and ensemble covariances.

Take a look at `/home/nonroot/shared/EDU/qgEnVar/yamls/3dvar_hybrid.yaml`. One thing you should notice immediately - it looks very similar to the 3dvar yaml from the **1.qg_3Dvar_tutorial**! In fact, much of the yaml file is indeed the same. For example, we are still using  ```cost type: 3D-Var```, and the `variational:` section is also the same as the 3DVar tutorial runs - the same minimizer and outer/inner loop strategies. 

The main difference in `3dvar_hybrid.yaml`with the original 3dvar yaml file (`qg3Dvar/yamls/3dvar_oneobs_default.yaml`) is pasted below:
```yaml
  background error:
    covariance model: hybrid            # activate hybrid covariance model mode
    components:  
    - covariance:                       
        covariance model: QgError        # Section that defines the static-B covariance model
        horizontal_length_scale: 2.2e6
        maximum_condition_number: 1.0e6
        standard_deviation: 1.8e7
        vertical_length_scale: 15000.0
      weight:
        value: 0.5                        # weight of static B covariances in hybrid-B
    - covariance:
        covariance model: ensemble        # Section that defines the ensemble covariances
        localization:                     # including localizations for the ensemble perturbations
          horizontal_length_scale: 8.0e6
          localization method: QG
          maximum_condition_number: 1.0e6
          standard_deviation: 1.0
          vertical_length_scale: 30000.0
        members from template:
          template:
            date: 2009-12-31T00:00:00Z
            filename: qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc 
          pattern: %mem%
          nmembers: 20
      weight:
        value: 0.5                        # weight of Ensemble covariances in hybrid-B
```

In the above, you can see that rather than defining the QGError as the single covariance model, we have activated the `hybrid` mode. And in doing so, multiple components of the covariance model can be defined. In the above, the first section defines the static-B, which uses the same `QGError` model and parameter settings as in the `3dvar_oneobs_default.yaml` experiment from the 3DVar tutorial. This includes the QgError static-B setting with a `horizontal_length_scale` of 2200 km and a `vertical_length_scale` of 15km, as was used for 3DVar experiments previously.

The second section defines the `ensemble` covarianes,  read in from files defined under `members from template`. The `localization` is also applied within this section, and for this experiment we have set it to a broad localization `8.0e6` (8000 km) as was used in the `dblloc` experiment in the LETKF tutorial. The broader value will help us analyze differences in static-ensemble weightings in B-matrix for #nVar

Finally, each covariance component contains a `weight` parameter that defines the weight of this component in the final hybrid covariances. Although each weight can be defined independently within JEDI, you should generally ensure that the sum of the weights add up to 1 for all components (in the above, there are two weights of 0.5 each, which do sum to 1).

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid.yaml 

There will be two files in `/home/nonroot/shared/EDU/qgEnVar/output/exp_default/da`. List them with the command below, do you know what each file represents?

In [ ]:
ls $JEDI_EDU/qgEnVar/output/exp_default/da

#### Experiment 2:  Static 0.9, ensemble 0.1

First, examine the file `qgEnVar/yamls/3dvar_hybrid_b90e10.yaml`. The diff command below summarizes the differences compared to `3dvar_hybrid.yaml` for the default experiment. Besides changes to the experiment directory for outputs, we have modified the `value:` parameter under the `weight:` section in the yaml that defines the weights for each component (static and ensemble) of the hybrid covariances.

In [ ]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid.yaml ./qgEnVar/yamls/3dvar_hybrid_b90e10.yaml

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_b90e10.yaml 

Check that you have analysis and increment files produced after the DA:

In [ ]:
ls $JEDI_EDU/qgEnVar/output/exp_b90e10/da

#### Experiment 3:  Static 0.1, Ensemble 0.9

Examine the file `qgEnVar/yamls/3dvar_hybrid_b10e90.yaml`. The diff command below summarizes the differences compared to 3dvar_hybrid.yaml for the default experiment. The changes are similar to experiment 2, except we are now assigning a weight of 0.9 to the ensemble covariances, and 0.1 to the static-B covariances.

In [6]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid.yaml ./qgEnVar/yamls/3dvar_hybrid_b10e90.yaml

23c23
<         value: 0.5                        # weight of static B covariances in hybrid-B
---
>         value: 0.1                        # weight of static B covariances in hybrid-B
39c39
<         value: 0.5                        # weight of Ensemble covariances in hybrid-B
---
>         value: 0.9                       # weight of Ensemble covariances in hybrid-B
50c50
<             obsfile: qgEnVar/output/exp_default/obs/3denvar_obsdataout.obs3d.nc
---
>             obsfile: qgEnVar/output/exp_b10e90/obs/3denvar_obsdataout.obs3d.nc
92c92
<       datadir: qgEnVar/output/exp_default/da
---
>       datadir: qgEnVar/output/exp_b10e90/da
99c99
<   datadir: qgEnVar/output/exp_default/da
---
>   datadir: qgEnVar/output/exp_b10e90/da


: 1

In [7]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_b10e90.yaml 

Configuration input file is: ./qgEnVar/yamls/3dvar_hybrid_b10e90.yaml
OOPS Starting 2026-04-29 23:43:01 (UTC+0000)
Full configuration is:YAMLConfiguration[path=./qgEnVar/yamls/3dvar_hybrid_b10e90.yaml, root={cost function => {cost type => 3D-Var , window begin => 2009-12-30T21:00:00Z , window length => PT6H , analysis variables => (x) , geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , background => {date => 2009-12-31T00:00:00Z , filename => qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc} , background error => {covariance model => hybrid , components => ({covariance => {covariance model => QgError , horizontal_length_scale => 2.2e+06 , maximum_condition_number => 1e+06 , standard_deviation => 1.8e+07 , vertical_length_scale => 15000} , weight => {value => 0.1}},{covariance => {covariance model => ensemble , localization => {horizontal_length_scale => 8e+06 , localization method => QG , maximum_condition_number => 1e+06 , standard_deviation => 1 , vertical_length_scale

Check that you have analysis and increment files produced after the DA:

In [8]:
ls $JEDI_EDU/qgEnVar/output/exp_b10e90/da

EnVar.an.2009-12-31T00:00:00Z.nc  EnVar.increment.in.2009-12-31T00:00:00Z.nc


#### Plot the results:
Finally, let's plot and compare the resulting analysis increments of each of these experiments. Note we are using the option `--plotstream` which overlays contours of streamfunction from the background field, to help assess flow-dependence.

In [9]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_default/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob_northeast.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_default/plots/EnVar_inc_b50e50 \
        --title "analysis increment, B 50% ens 50%" --fieldmax 6e7
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_b10e90/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob_northeast.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_b10e90/plots/EnVar_inc_b10e90 \
        --title "analysis increment, B 10% ens 90%" --fieldmax 6e7
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_b90e10/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob_northeast.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_b90e10/plots/EnVar_inc_b90e10 \
        --title "analysis increment, B 90% ens 10%" --fieldmax 6e7

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared/EDU/qgEnVar/output/exp_default/da/EnVar.an.2009-12-31T00:00:00Z.nc
 - basefilepath: /home/nonroot/shared/EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc
 - fieldmax: 6e7
 - scalemax: False
 - plotObsLocations: None
 - plotObsValues: /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob_northeast.obs3d.nc
 - plothofx: False
 - plotwind: False
 - plotstream: True
 - gif: None
 - output: /home/nonroot/shared/EDU/qgEnVar/output/exp_default/plots/EnVar_inc_b50e50
 - title: analysis increment, B 50% ens 50%
Run script
['/home/nonroot/shared/EDU/qgEnVar/output/exp_default/da/EnVar.an.2009-12-31T00:00:00Z.nc']
lon,lat,val of first ob = 150.0,50.0,1388457.7714409828
Variable x, RMSD = 11199774.652808351
data range:(-66386495.466908395,17158833.454286993), plot range: (-60000000.0,60000000.0) 
 -> plot produced: /home/nonroot/shared/EDU/qgEnVar/output/exp_default/plots/EnVar_inc_b50e50_x_diff.jpg
lon,lat

Examine the produced plots (copied below as well). 
- Can you see the effects underlying hybrid covariances have on the resulting increments?
- How do the increments change as the proportion of hybrid-B increases with the ensemble covariances (and decreases the static B)? 

<center><img src="images/EnVar_inc_b90e10_x_diff.jpg" width="700"/></center>
<center><img src="images/EnVar_inc_b50eb50_x_diff.jpg" width="700"/></center>
<center><img src="images/EnVar_inc_b10e90_x_diff.jpg" width="700"/></center>

## Experiments 4-6: Assimilate with Multiple Observations, varying the weights of static and ensemble covariances

---

Verify you have the `truth_100obs.obs3d.nc` file from the `qgstart` tutorial:

In [ ]:
cd $JEDI_EDU
ls ./qgstart/output/obs/*nc

Comparing the multiple obs experiment to the default single-obs experiment, you will see all we have changed is the observation file (obsfile) within the `obsdatain` section:

In [ ]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid.yaml ./qgEnVar/yamls/3dvar_hybrid_mult_b50e50.yaml

Let's run this experiment now:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_mult_b50e50.yaml 

Verify that the above run executed correctly, listing the contents of the `da` directory. There should be two files produced:

In [ ]:
ls $JEDI_EDU/qgEnVar/output/exp_mult_b50e50/da

As in the single-observation experiment, we will run two additional configurations varying the weights of static B and ensemble covariance terms in the hybrid covariances. We will test one run with mostly ensemble covariances (static 10%, ensemble 90%), and another run with mostly static covariances from the QGError model (static 90%, ensemble 10%). 

You can verify these settings for each experiment looking at the below `diff` commands:

In [ ]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid_mult_b50e50.yaml ./qgEnVar/yamls/3dvar_hybrid_mult_b10e90.yaml

In [ ]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid_mult_b50e50.yaml ./qgEnVar/yamls/3dvar_hybrid_mult_b90e10.yaml

Now let's run each 3dEnVar experiment, verifying at the end both produced results:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_mult_b10e90.yaml 

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_mult_b90e10.yaml 

In [ ]:
echo "exp_mult_b10e90 output:"
ls -1 $JEDI_EDU/qgEnVar/output/exp_mult_b10e90/da
echo -e "\nexp_mult_b90e10 output:"
ls -1  $JEDI_EDU/qgEnVar/output/exp_mult_b90e10/da

<br>Before you look at the results, take a moment to think and form an idea of what you expect to see. After that, have a look at the results via the increment figures, generated via:

In [ ]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_mult_b50e50/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_mult_b50e50/plots/EnVar_inc_mult_b50eb50 \
        --title "analysis increment, B 50% ens 50%"
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_mult_b10e90/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_mult_b10e90/plots/EnVar_inc_mult_b10e90 \
        --title "analysis increment, B 10% ens 90%"
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_mult_b90e10/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_mult_b90e10/plots/EnVar_inc_mult_b90e10 \
        --title "analysis increment, B 90% ens 10%"

<center><img src="images/envar_mult_inc.gif" width="700"/></center>
A note on the above plots: we have included the option `--plotstream`, which will plot  countour lines of streamfunction from the second file given for each plot (in this case, the background streamfunction). This can help to assess flow dependence when looking at increment figures.


In [ ]:
# Compare meananl and meanbkg errors
cd $JEDI_EDU/plots_scripts
for exp in b50e50 b10e90 b90e10; do
  python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_mult_$exp/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/EnVar_anl_error \
        --title "Control analysis error" \
        >& $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/plot_anl_err.log
  [[ $? -eq 0 ]] && echo SUCCESS PLOT ANL ERROR $exp || echo ERROR! ANL plot $exp
  python ./plot.py qg fields \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/EnVar_bkg_error \
        --title "Control background error" \
        >& $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/plot_bkg_err.log
  [[ $? -eq 0 ]] && echo SUCCESS PLOT BKG ERROR $exp || echo ERROR! BKG plot $exp
done

![mult_err](images/3denvar_mult_err.png)

A couple questions to think about as you examine the above figures of analysis increments and errors:

- Can you see the effect of flow dependence in this case? Is it obvious, or more difficult to tell than the single-ob experiment?
- Can you see flow-dependence increase in the increments as the ensemble percentage increases?
- Which experiment produces the lowest RMS analysis error?

For the second question, it is almost impossible to tell visually, so we will need to compute the RMSE directly. We have provided a python script `compute_rmse_layers.py`.  To run this script, you need to provide it the model file to evaluate (either background or analysis), the truth file to verify against, which variable to compute RMSE for (x,u,v,q), and the destination text file to output the results into, i.e.,

> python compute_rmse_layers.py <input_model.nc> <input_truth.nc> -v x -o <output_rmse_file.txt> -l \<line_label_in_output_text_file>

In [14]:
cd $JEDI_EDU
output_filename="./qgEnVar/output/rmse_stream.txt"

# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

echo "Computing background rmse . . ."
# Run for background mean file
python ./plots_scripts/compute_rmse_layers.py \
          ./qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc  \
          ./qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc \
          -v x -o $output_filename -l "bg"


for exp in b90e10 b50e50 b10e90; do
   echo "Computing analysis rmse for $exp . . ."

   # Run for analysis  file
   python ./plots_scripts/compute_rmse_layers.py \
          ./qgEnVar/output/exp_mult_${exp}/da/EnVar.an.2009-12-31T00:00:00Z.nc \
          ./qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc \
          --v x -o $output_filename -l "an ${exp}"
done
echo ""
cat $output_filename

removed './qgEnVar/output/rmse_stream.txt'
Computing background rmse . . .
RMSE written to ./qgEnVar/output/rmse_stream.txt
Computing analysis rmse for b90e10 . . .
RMSE written to ./qgEnVar/output/rmse_stream.txt
Computing analysis rmse for b50e50 . . .
RMSE written to ./qgEnVar/output/rmse_stream.txt
Computing analysis rmse for b10e90 . . .
RMSE written to ./qgEnVar/output/rmse_stream.txt

bg Layer 0: RMSE = 29406041.003888
bg Layer 1: RMSE = 32413972.444049
an b90e10 Layer 0: RMSE = 10178308.027330
an b90e10 Layer 1: RMSE = 10413381.658743
an b50e50 Layer 0: RMSE = 8619737.166723
an b50e50 Layer 1: RMSE = 9305803.115895
an b10e90 Layer 0: RMSE = 8533778.381745
an b10e90 Layer 1: RMSE = 8936736.275786


## Summary and Conclusions

In this tutorial, we have explored Three-Dimensional Ensemble-Variational data assimilation (3DEnVar), a hybrid method that combines the strengths of variational and ensemble data assimilation approaches. The key concepts we covered include:

- **Motivation for Hybrid Methods:** 3DEnVar combines the strengths from both the variational and ensemble approaches.

- **Hybrid Background Error Covariance:** The hybrid covariance is effectively a weighted combination of static and ensemble-derived components: $\mathbf{B}_{hybrid} = \alpha \mathbf{B}_{static} + (1-\alpha) (\mathcal{L} \circ \mathbf{B}_{ens})$. The weights $\alpha$ and $(1-\alpha)$ control the relative contributions of each component.


Hybrid data assimilation methods have been utilized by multiple operational numerical weather prediction centers worldwide. They provide a practical framework for incorporating flow-dependent error information while retaining the flexibility and quality control capabilities of variational methods. As you continue your studies in data assimilation, you will encounter many extensions and refinements of the hybrid approach, including four-dimensional variants (4DEnVar) and advanced localization techniques.